# dsaMixer connection check

Run all cells (kernel: this project's **`.venv`** Python) to check every external connection the app uses:

| # | Connection | Used for | Cost / side effects |
|---|---|---|---|
| 1 | `.env` variables | everything below | reads names only, never prints values |
| 2 | Postgres (`DATABASE_URL`) | all app data | read-only queries |
| 3 | Gmail SMTP | morning/evening emails | logs in and out, **sends nothing** unless you enable cell 3b |
| 4 | OpenRouter | weekly AI planner | key-info call, free |
| 5 | Anthropic API | planner fallback (only if `ANTHROPIC_API_KEY` is set) | model-list call, free |
| 6 | Local board server | the UI at http://127.0.0.1:8765 | one GET request |
| 7 | Claude Code CLI + launchd jobs | homework agents and scheduled runs | reads status only |

Everything except the app's own database is optional. A connection you haven't set up is reported as **SKIP**, never FAIL. Each check records **PASS / FAIL / SKIP**, and the last cell prints a summary. No passwords, keys or URLs are printed.

In [ ]:
import os, sys, json, subprocess, smtplib, urllib.request, urllib.error
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "dsamixer").is_dir() and ROOT != ROOT.parent:  # find the project root from wherever the notebook runs
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dsamixer.core import config, db
db.load_env()

RESULTS = {}

def record(name, status, detail=""):
    RESULTS[name] = (status, detail)
    print(f"{status:4}  {name}" + (f"  —  {detail}" if detail else ""))

def http_json(url, headers=None, timeout=15):
    req = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, json.loads(r.read() or b"null")

print("project root:", ROOT)

## 1. `.env` variables (names only)

In [ ]:
groups = {
    "Postgres": ["DATABASE_URL"],
    "Email": ["SMTP_USER", "SMTP_PASSWORD", "MAIL_TO"],
    "AI planner": ["OPENROUTER_API_KEY", "ANTHROPIC_API_KEY"],
}
print(".env file:", "found" if db.ENV_PATH.exists() else "not present (optional)")
for group, keys in groups.items():
    print(f"\n{group}:")
    for k in keys:
        print(f"  {'set    ' if os.environ.get(k) else 'missing'}  {k}")
record("env file", "PASS" if db.ENV_PATH.exists() else "SKIP",
       "" if db.ENV_PATH.exists() else "no .env: fine, every setting in it is optional (copy .env.example to add some)")

## 2. Postgres

In [ ]:
url = os.environ.get("DATABASE_URL")
if not url:
    record("postgres", "SKIP", "DATABASE_URL not set (app uses SQLite)")
else:
    try:
        import psycopg
        with psycopg.connect(url, connect_timeout=5) as c:
            name = c.execute("SELECT current_database()").fetchone()[0]
            version = c.info.server_version
            tables = {r[0] for r in c.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='public'")}
        missing = [t for t in db.TABLES if t not in tables]
        record("postgres", "PASS" if not missing else "FAIL",
               f"db '{name}', server {version // 10000}.{version % 10000}" + (f", missing tables: {missing}" if missing else ""))
    except Exception as e:
        record("postgres", "FAIL", f"{type(e).__name__}: {str(e).splitlines()[0][:150]}")

In [ ]:
# The app's own database layer: which backend it picks, plus row counts.
try:
    conn = db.connect()
    db.init(conn)  # the same idempotent setup the app runs at start-up, so a first run works too
    counts = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in db.TABLES}
    conn.close()
    print("backend:", db.backend())
    for t, n in counts.items():
        print(f"  {t:15} {n}")
    ok = counts["questions"] == 507 and counts["progress"] == 507
    record("app db layer", "PASS" if ok else "FAIL", f"{db.backend()}, {counts['tickets']} tickets"
           + ("" if ok else ", expected 507 questions/progress"))
except Exception as e:
    record("app db layer", "FAIL", f"{type(e).__name__}: {str(e)[:150]}")

## 3. Gmail SMTP (login only, no email sent)

In [ ]:
user, pw = os.environ.get("SMTP_USER"), os.environ.get("SMTP_PASSWORD")
host, port = os.environ.get("SMTP_HOST", "smtp.gmail.com"), int(os.environ.get("SMTP_PORT", 465))
if not (user and pw and os.environ.get("MAIL_TO")):
    record("smtp", "SKIP", "SMTP_USER / SMTP_PASSWORD / MAIL_TO not all set")
else:
    try:
        with smtplib.SMTP_SSL(host, port, timeout=15) as s:
            s.login(user, pw)
        record("smtp", "PASS", f"logged in to {host}:{port}")
    except smtplib.SMTPAuthenticationError:
        record("smtp", "FAIL", "login rejected: use a Gmail App Password, not your normal password")
    except smtplib.SMTPServerDisconnected:
        # Gmail dropped the connection at login instead of answering "wrong password".
        record("smtp", "FAIL", "Gmail closed the connection at login. Check: SMTP_USER is the exact personal "
               "@gmail.com account the App Password was made for; 2-Step Verification is on and the App "
               "Password still exists; the .env lines have no quotes or trailing comments; a work "
               "(Google Workspace) account may block SMTP logins")
    except Exception as e:
        record("smtp", "FAIL", f"{type(e).__name__}: {str(e)[:150]}")

### 3b. Optional: send a real test email
Set `SEND_TEST_EMAIL = True` and run this cell to send one email to `MAIL_TO`.

In [ ]:
SEND_TEST_EMAIL = False

if SEND_TEST_EMAIL:
    from dsamixer.integrations import notify
    try:
        notify.send("dsaMixer test email", "If you can read this, the morning/evening emails will work.")
        record("smtp send", "PASS", "test email sent to MAIL_TO")
    except BaseException as e:
        record("smtp send", "FAIL", f"{type(e).__name__}: {str(e)[:150]}")
else:
    print("skipped (SEND_TEST_EMAIL = False)")

## 4. OpenRouter (weekly AI planner)

In [ ]:
key = os.environ.get("OPENROUTER_API_KEY")
if not key:
    record("openrouter", "SKIP", "OPENROUTER_API_KEY not set")
else:
    try:
        _, info = http_json("https://openrouter.ai/api/v1/key", {"Authorization": f"Bearer {key}"})
        d = info.get("data", {})
        limit = d.get("limit")
        remaining = d.get("limit_remaining")
        credit = "no spending limit" if limit is None else f"${remaining:.2f} of ${limit:.2f} left"
        record("openrouter key", "PASS", f"usage so far ${d.get('usage', 0):.4f}, {credit}")
    except urllib.error.HTTPError as e:
        record("openrouter key", "FAIL", f"HTTP {e.code}: " + ("invalid key" if e.code == 401 else e.reason))
    except Exception as e:
        record("openrouter key", "FAIL", f"{type(e).__name__}: {str(e)[:150]}")

    # Is the configured model still offered?
    try:
        from dsamixer.integrations import ai_planner
        model = ai_planner.model_name(config.load(db.connect()))  # OPENROUTER_MODEL from .env if set
        _, models = http_json("https://openrouter.ai/api/v1/models")
        found = next((m for m in models["data"] if m["id"] == model), None)
        if not found:
            record("openrouter model", "FAIL", f"{model} not listed on OpenRouter")
        else:
            structured = "structured_outputs" in (found.get("supported_parameters") or [])
            record("openrouter model", "PASS" if structured else "FAIL",
                   f"{model}" + ("" if structured else " doesn't support structured outputs"))
    except Exception as e:
        record("openrouter model", "FAIL", f"{type(e).__name__}: {str(e)[:150]}")

## 5. Anthropic API (only if `ANTHROPIC_API_KEY` is set)

In [ ]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    record("anthropic", "SKIP", "ANTHROPIC_API_KEY not set (OpenRouter is used instead)")
else:
    try:
        import anthropic
        model = config.load(db.connect())["ai_planner"]["model"]
        m = anthropic.Anthropic().models.retrieve(model)
        record("anthropic", "PASS", f"key works, {m.id} available")
    except Exception as e:
        record("anthropic", "FAIL", f"{type(e).__name__}: {str(e)[:150]}")

## 6. Local board server

In [ ]:
try:
    _, s = http_json("http://127.0.0.1:8765/api/stats", timeout=5)
    record("board server", "PASS", f"running, {s['done']}/{s['total']} questions done")
except Exception:
    record("board server", "SKIP", "not running on :8765 (start it with ./run.sh, or pass --port if you use another)")

## 7. Claude Code CLI and scheduled jobs

In [ ]:
try:
    v = subprocess.run(["claude", "--version"], capture_output=True, text=True, timeout=20).stdout.strip()
    record("claude cli", "PASS" if v else "FAIL", v or "no version output")
except FileNotFoundError:
    record("claude cli", "SKIP", "`claude` not on PATH (only needed for the homework agents)")

settings = ROOT / ".claude" / "settings.json"
if settings.exists():
    allow = json.loads(settings.read_text()).get("permissions", {}).get("allow", [])
    need = {"Bash(./run.sh *)", "Edit(/homework/**/STATUS.md)"}
    record("agent permissions", "PASS" if need <= set(allow) else "FAIL",
           "settings.json allows run.sh + STATUS.md edits" if need <= set(allow) else f"missing: {need - set(allow)}")
else:
    record("agent permissions", "SKIP", ".claude/settings.json not present (only needed for scheduled headless homework sync; see docs/guides/homework-agents.md)")

jobs = ["com.dsamixer.server", "com.dsamixer.morning", "com.dsamixer.evening", "com.dsamixer.homework-sync"]
if sys.platform != "darwin":
    record("launchd jobs", "SKIP", "launchd is macOS only (see docs/guides/integrations.md for cron)")
else:
    listed = subprocess.run(["launchctl", "list"], capture_output=True, text=True).stdout
    loaded = [j for j in jobs if j in listed]
    record("launchd jobs", "PASS" if len(loaded) == len(jobs) else "SKIP",
           f"{len(loaded)}/{len(jobs)} loaded" + ("" if loaded else ": run ./run.sh schedule install to enable"))

## Summary

In [ ]:
icons = {"PASS": "✅", "FAIL": "❌", "SKIP": "⏭️"}
for name, (status, detail) in RESULTS.items():
    print(f"{icons[status]} {status:4}  {name:18} {detail}")
fails = [n for n, (s, _) in RESULTS.items() if s == "FAIL"]
print("\nAll good." if not fails else f"\n{len(fails)} failing: {', '.join(fails)}")